# Task 3 perturbation-response playground

Task 3 is easy to misread because a mutant can look globally similar to WT while still having a very specific response that matters for scoring. This notebook starts from the public matched WT/Mab21l2 pair and deliberately changes only the mean perturbation response.

These are target-aware teaching controls, not held-out predictions. The transformations are defined once in [`experiments.py`](experiments.py) and reused by the committed result generator.

In [ ]:
!pip -q install "git+https://github.com/aristoteleo/veckit.git@46d41e63f42a9aab815db20b742feeccd249cb17" pandas matplotlib

In [ ]:
from pathlib import Path
import sys, urllib.request
import anndata as ad
import pandas as pd

LAB = Path.cwd() / 'intuition-lab'
if not (LAB / 'experiments.py').exists(): LAB = Path.cwd()
sys.path.insert(0, str(LAB))
from experiments import make_t3_failures, score_t3_failures

DATA = Path('/content/vec_t3_failures')
DATA.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/aristoteleo/veckit/46d41e63f42a9aab815db20b742feeccd249cb17/data/'
for name in ['sample_wt.h5ad', 'sample_mab21l2_ko.h5ad']:
    p = DATA / name
    if not p.exists(): urllib.request.urlretrieve(base + name, p)

wt_path = DATA / 'sample_wt.h5ad'
ko_path = DATA / 'sample_mab21l2_ko.h5ad'
wt, ko = ad.read_h5ad(wt_path), ad.read_h5ad(ko_path)
print('WT:', wt.shape, '| KO:', ko.shape)

## Change only the response

Let the known public mean response be `delta = mean(KO) - mean(WT)`. We create predictions with 0%, 25%, 50%, 100%, 150%, and 200% of that response, plus a reversed response and a gene-shuffled response.

The interesting comparison is between absolute similarity to the mutant and whether the **change from WT** points in the right direction with the right magnitude.

In [ ]:
preds, meta = make_t3_failures(wt, ko, DATA / 'predictions', seed=0)
print('alphas:', meta['alphas'])
print('built:', list(preds))

In [ ]:
scores = score_t3_failures(preds, target=ko_path, wt=wt_path)
scores[['de_score', 'de_direction', 'severity_slope', 'mmd_u', 'variogram', 'absolute_pb_pearson']].round(4)

In [ ]:
sweep = scores.loc[[f'alpha_{a:g}' for a in meta['alphas']], ['de_score', 'de_direction', 'severity_slope', 'absolute_pb_pearson']].copy()
sweep.index = meta['alphas']
sweep.index.name = 'response scale alpha'
sweep.round(4)

## What I would remember

The zero-response control is the useful sanity check. It can retain high absolute pseudobulk correlation simply because WT and KO share a lot of baseline expression, while completely missing the knockout effect. Direction and magnitude are separate failure modes too: getting the right genes to move the right way does not guarantee that the response is strong enough.